# DDL: `dbspend360_pipeline_dbu_cost`

Creates the per-pipeline / per-day / per-cluster / per-product Databricks DBU cost staging table populated by
`dbspend360_pipeline_dbu_cost_app`.

Sibling of `dbspend360_pool_dbu_cost`, but keyed on `(workspace_id, pipeline_id, usage_date, cluster_id, billing_origin_product)`.
Filters `system.billing.usage` to `usage_metadata.dlt_pipeline_id IS NOT NULL` (the canonical declarative-pipeline filter —
captures DLT, DBSQL materialized views / streaming tables, online tables, vector search, model serving, and AI functions,
for BOTH serverless and classic compute).

`workspace_id` is part of the key because `pipeline_id` is only unique within a single workspace. `cluster_id` is **NULL for
serverless** (the serverless signal) and is kept — not coalesced to a sentinel — so the v2 cloud-cost join (which is
`cluster_id`-keyed) needs no re-ingest, and the staging MERGE matches `cluster_id` with null-safe equality (`<=>`).
`billing_origin_product` stays in the grain so the per-workload `$` split is exact downstream. `update_cost` /
`maintenance_cost` are split out now (cheap, since both sub-ids are already read) for a v2 maintenance-ratio KPI; note their
sum need not equal `databricks_cost` (a row can carry neither sub-id).

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_pipeline_dbu_cost (
  workspace_id            STRING,
  pipeline_id             STRING,
  usage_date              DATE,
  cluster_id              STRING,
  billing_origin_product  STRING,
  compute_mode            STRING,
  databricks_cost         DOUBLE,
  update_cost             DOUBLE,
  maintenance_cost        DOUBLE,
  currency                STRING,
  sku_name                STRING,
  created_at              TIMESTAMP,
  updated_at              TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_pipeline_dbu_cost")